# Building a STAC catalog from scratch

**STAC** (SpatioTemporal Asset Catalog) is a common language for describing
geospatial data on the web.  Almost every modern EO data archive -- Landsat,
Sentinel, commercial providers -- exposes data through a STAC API.

The hierarchy is simple:

```
Catalog
  Collection  (a dataset, e.g. Sentinel-2 L2A)
    Item      (one scene, one granule)
      Asset   (an actual file: a band GeoTIFF, a thumbnail, ...)
```

In this notebook we:
1. Define metadata for two real Sentinel-2 scenes
2. Build the full STAC object tree with `pystac`
3. Validate everything against the STAC spec
4. Save it as static JSON files
5. Serve it on `localhost` and open it in STAC Browser


## 1. Install and import


In [ ]:
%pip install -q 'pystac[validation]>=1.10' jsonschema


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, threading, functools, webbrowser

import pystac
from pystac import (
    Catalog, CatalogType,
    Collection, Extent, SpatialExtent, TemporalExtent,
    Item, Asset, MediaType, Link,
)

# Catalog files will be written here (relative to this notebook)
CATALOG_DIR = Path('../catalog').resolve()
CATALOG_DIR.mkdir(parents=True, exist_ok=True)

PORT = 8080
print(f'pystac {pystac.__version__}')
print(f'Catalog will be saved to: {CATALOG_DIR}')


## Example Sentinel-2 scene metadata

Two summer scenes over Tuscany (UTM tile 32TNT).  Assets point to real
public Cloud-Optimised GeoTIFFs on the `sentinel-cogs` S3 bucket.
No download is needed -- the URLs are embedded as links in the STAC JSON.


In [ ]:
BASE = 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs'

# Bounding box: approximate extent of UTM tile 32TNT (WGS84)
BBOX_32TNT = [9.76, 43.26, 11.52, 44.27]

# Polygon geometry matching the bbox
def bbox_to_geom(bbox):
    w, s, e, n = bbox
    return {
        'type': 'Polygon',
        'coordinates': [[
            [w, s], [e, s], [e, n], [w, n], [w, s]
        ]]
    }

SCENES = [
    {
        'id':           'S2A_32TNT_20240601_0_L2A',
        'datetime':     datetime(2024, 6, 1, 10, 16, 23, tzinfo=timezone.utc),
        'platform':     'sentinel-2a',
        'cloud_cover':  3.2,
        'bbox':         BBOX_32TNT,
        'base_url':     f'{BASE}/32/T/NT/2024/6/S2A_32TNT_20240601_0_L2A',
    },
    {
        'id':           'S2B_32TNT_20240612_0_L2A',
        'datetime':     datetime(2024, 6, 12, 10, 14, 55, tzinfo=timezone.utc),
        'platform':     'sentinel-2b',
        'cloud_cover':  8.7,
        'bbox':         BBOX_32TNT,
        'base_url':     f'{BASE}/32/T/NT/2024/6/S2B_32TNT_20240612_0_L2A',
    },
]

print(f'{len(SCENES)} scenes defined')
for s in SCENES:
    print(f"  {s['id']}  {s['datetime'].date()}  cloud={s['cloud_cover']}%")


## Build the STAC object tree

We work bottom-up: Assets go into Items, Items into a Collection,
the Collection into the Catalog.


In [ ]:
# ── Catalog ──────────────────────────────────────────────────────────────
catalog = Catalog(
    id='sentinel-2-demo',
    description='Demo STAC catalog built with pystac. Two Sentinel-2 L2A scenes over Tuscany.',
    title='Sentinel-2 Demo',
)

# ── Collection ───────────────────────────────────────────────────────────
# Extent covers all items we add below
spatial_extent  = SpatialExtent(bboxes=[BBOX_32TNT])
temporal_extent = TemporalExtent(intervals=[
    [SCENES[0]['datetime'], SCENES[-1]['datetime']]
])

collection = Collection(
    id='sentinel-2-l2a',
    description='Sentinel-2 Level-2A surface reflectance (demo subset)',
    title='Sentinel-2 L2A',
    extent=Extent(spatial_extent, temporal_extent),
    license='proprietary',
    extra_fields={
        'platform':      'sentinel-2',
        'constellation': 'sentinel-2',
        'instruments':   ['msi'],
    },
)

# ── Items and Assets ─────────────────────────────────────────────────────
for scene in SCENES:
    item = Item(
        id=scene['id'],
        geometry=bbox_to_geom(scene['bbox']),
        bbox=scene['bbox'],
        datetime=scene['datetime'],
        properties={
            'platform':        scene['platform'],
            'constellation':   'sentinel-2',
            'instruments':     ['msi'],
            'eo:cloud_cover':  scene['cloud_cover'],
            'processing:level':'L2A',
        },
    )

    base = scene['base_url']

    # Thumbnail (JPEG overview)
    item.add_asset('thumbnail', Asset(
        href=f'{base}/thumbnail.jpg',
        media_type=MediaType.JPEG,
        title='Thumbnail',
        roles=['thumbnail'],
    ))

    # True-colour composite (10 m)
    item.add_asset('visual', Asset(
        href=f'{base}/TCI.tif',
        media_type=MediaType.COG,
        title='True colour (TCI, 10 m)',
        roles=['visual'],
    ))

    # Red band B04 (10 m)
    item.add_asset('red', Asset(
        href=f'{base}/B04.tif',
        media_type=MediaType.COG,
        title='Red - B04 (10 m)',
        roles=['data'],
        extra_fields={'eo:bands': [{'name': 'B04', 'common_name': 'red',  'center_wavelength': 0.665}]},
    ))

    # Near-infrared band B08 (10 m)
    item.add_asset('nir', Asset(
        href=f'{base}/B08.tif',
        media_type=MediaType.COG,
        title='NIR - B08 (10 m)',
        roles=['data'],
        extra_fields={'eo:bands': [{'name': 'B08', 'common_name': 'nir', 'center_wavelength': 0.842}]},
    ))

    collection.add_item(item)

catalog.add_child(collection)

# Quick summary
print(f'Catalog : {catalog.id}')
print(f'Collection: {collection.id}')
for item in collection.get_items():
    print(f'  Item: {item.id}  |  assets: {list(item.assets.keys())}')


## Validate against the STAC spec

`pystac` can check each object against the official JSON schemas.
Any missing required field or wrong type shows up here before we write files.


In [ ]:
print('Validating items...')
for item in collection.get_items():
    issues = item.validate()
    status = 'OK' if issues else 'OK'
    print(f'  {item.id}: {status}')

print('Validating full catalog...')
num = catalog.validate_all()
print(f'  {num} objects validated, no errors.')


## Save as static JSON

`normalize_hrefs` assigns a consistent local file path to every object,
then `save` writes them out.  `SELF_CONTAINED` means each JSON file
carries absolute `self` links -- easy to serve from any directory.


In [ ]:
import shutil

# Start fresh so re-running the cell doesn't leave stale files
if CATALOG_DIR.exists():
    shutil.rmtree(CATALOG_DIR)
CATALOG_DIR.mkdir(parents=True)

catalog.normalize_hrefs(str(CATALOG_DIR))
catalog.save(catalog_type=CatalogType.SELF_CONTAINED)

# Show what was written
files = sorted(CATALOG_DIR.rglob('*.json'))
print(f'Written {len(files)} JSON files to {CATALOG_DIR}:')
for f in files:
    print(f'  {f.relative_to(CATALOG_DIR)}')

# Pretty-print the root catalog to see its structure
print('\n--- catalog.json ---')
root = json.loads((CATALOG_DIR / 'catalog.json').read_text())
print(json.dumps(root, indent=2))


## Serve the catalog on localhost

A minimal HTTP server with CORS headers so that the hosted STAC Browser
(which runs on `https://`) can fetch from your `http://localhost`.
The server runs in a background thread and stays alive for the
duration of the kernel session.


In [ ]:
from http.server import HTTPServer, SimpleHTTPRequestHandler

class CORSHandler(SimpleHTTPRequestHandler):
    """Adds CORS headers so external browser tools can fetch from localhost."""
    def end_headers(self):
        self.send_header('Access-Control-Allow-Origin',  '*')
        self.send_header('Access-Control-Allow-Headers', '*')
        self.send_header('Cache-Control', 'no-store')
        super().end_headers()
    def log_message(self, *args):
        pass  # silence request log

# Serve files from the catalog directory
handler = functools.partial(CORSHandler, directory=str(CATALOG_DIR))

try:
    server = HTTPServer(('localhost', PORT), handler)
    thread = threading.Thread(target=server.serve_forever, daemon=True)
    thread.start()
    print(f'Server running on http://localhost:{PORT}/')
except OSError:
    print(f'Port {PORT} already in use -- server may already be running.')

# Verify root catalog is reachable
import urllib.request
try:
    with urllib.request.urlopen(f'http://localhost:{PORT}/catalog.json') as r:
        data = json.loads(r.read())
    print(f'Catalog root reachable: id = "{data["id"]}"')
except Exception as e:
    print(f'Could not reach catalog: {e}')


## Open in STAC Browser

The Radiant Earth STAC Browser is a hosted web app that can point at any
STAC catalog URL -- including your localhost one.

> If your browser blocks mixed content (HTTPS page fetching HTTP),
> open the URL manually and allow it, or use Firefox which prompts you.


In [ ]:
ROOT_URL    = f'http://localhost:{PORT}/catalog.json'
BROWSER_URL = f'https://radiantearth.github.io/stac-browser/#/external/{ROOT_URL}'

print('Catalog root :', ROOT_URL)
print('STAC Browser :', BROWSER_URL)

# Open STAC Browser in default system browser
webbrowser.open(BROWSER_URL)

# Also embed it in the notebook output for convenience
from IPython.display import IFrame, display
display(IFrame(BROWSER_URL, width='100%', height=600))


## Explore the raw JSON

You can also inspect each file directly.  Here we pretty-print one Item
to see exactly what STAC JSON looks like on disk.


In [ ]:
# Print the first item JSON
item_files = sorted(CATALOG_DIR.rglob('*.json'))
item_files = [f for f in item_files if 'S2' in f.name]  # skip collection/catalog

if item_files:
    item_json = json.loads(item_files[0].read_text())
    print(f'--- {item_files[0].name} ---')
    print(json.dumps(item_json, indent=2, default=str))
